# Bài 61: Kết xuất thần kinh (Neural Rendering)

Cấu trúc từ chuyển động ở Bài 29 khôi phục một đám mây điểm 3D thưa từ nhiều ảnh. **Kết xuất thần kinh (neural rendering)** — nổi tiếng nhất là **NeRF** (<a href="../references.html#mildenhall-2020-nerf">Mildenhall et al., 2020</a>) — đòi hỏi thứ gì đó phong phú hơn: một biểu diễn *liên tục* của toàn bộ một cảnh 3D, đủ dày đặc để kết xuất một ảnh chân thực như ảnh chụp từ bất kỳ góc nhìn camera nào, kể cả những góc chưa bao giờ được quan sát trong khi huấn luyện. Biểu diễn đó hoàn toàn không phải là một lưới (mesh) hay một đám mây điểm; nó là trọng số của một mạng nơ-ron nhỏ. Bài này xây dựng một mạng như vậy từ đầu: một mạng tọa độ được huấn luyện hoàn toàn bằng cách so sánh các điểm ảnh được kết xuất với các điểm ảnh thật, không có giám sát 3D nào ở bất cứ đâu trong hàm mất mát.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

## Dữ liệu chuẩn: một bộ dò tia cổ điển cho một cảnh tổng hợp (Ground truth: a classical ray-tracer for a synthetic scene)
Một hình cầu có đổ bóng duy nhất, được chụp từ 8 vị trí camera xếp thành một vòng quanh nó (mô hình camera lỗ kim và sự hình thành ảnh ở Bài 26). Màu của mỗi điểm ảnh đến từ một phép giao tia-hình cầu cổ điển, không học được với đổ bóng theo hướng đơn giản &mdash; điều này *chỉ* được dùng để tạo dữ liệu huấn luyện và một góc nhìn chuẩn được giữ lại để chấm điểm; bản thân mạng không bao giờ thấy vị trí, bán kính hay màu thực của hình cầu.

In [ ]:
IMG = 16
SPHERE_CENTER = np.array([0.0, 0.0, 0.0])
SPHERE_RADIUS = 1.0
SPHERE_COLOR = np.array([1.0, 0.3, 0.2])

def look_at(cam_pos, target=np.array([0, 0, 0]), up=np.array([0, 1, 0])):
    z = (cam_pos - target); z = z / np.linalg.norm(z)
    x = np.cross(up, z); x = x / np.linalg.norm(x)
    y = np.cross(z, x)
    return np.stack([x, y, z], axis=1)

def get_rays(cam_pos, R, img_size=IMG, fov=60.0):
    f = img_size / (2 * np.tan(np.radians(fov) / 2))
    ys, xs = np.meshgrid(np.arange(img_size), np.arange(img_size), indexing='ij')
    dirs_cam = np.stack([(xs - img_size/2 + 0.5) / f, -(ys - img_size/2 + 0.5) / f,
                          -np.ones_like(xs, dtype=np.float64)], axis=-1)
    dirs_world = dirs_cam @ R.T
    dirs_world = dirs_world / np.linalg.norm(dirs_world, axis=-1, keepdims=True)
    origins = np.broadcast_to(cam_pos, dirs_world.shape)
    return origins.astype(np.float32), dirs_world.astype(np.float32)

def analytic_render(cam_pos):
    R = look_at(cam_pos)
    origins, dirs = get_rays(cam_pos, R)
    oc = origins - SPHERE_CENTER
    a = np.sum(dirs * dirs, axis=-1)
    b = 2 * np.sum(oc * dirs, axis=-1)
    c = np.sum(oc * oc, axis=-1) - SPHERE_RADIUS ** 2
    disc = b ** 2 - 4 * a * c
    hit = disc > 0
    t = np.where(hit, (-b - np.sqrt(np.clip(disc, 0, None))) / (2 * a), np.inf)
    hit = hit & (t > 0)
    img = np.zeros((IMG, IMG, 3), dtype=np.float32)
    hit_pos = origins + np.where(hit, t, 0)[..., None] * dirs
    normal = (hit_pos - SPHERE_CENTER) / SPHERE_RADIUS
    light_dir = np.array([1.0, 1.0, 1.0]); light_dir = light_dir / np.linalg.norm(light_dir)
    shade = np.clip(np.sum(normal * light_dir, axis=-1), 0.2, 1.0)
    img[hit] = (SPHERE_COLOR[None, :] * shade[..., None])[hit]
    return img, origins, dirs, hit

n_train_views = 8
angles = np.linspace(0, 2 * np.pi, n_train_views, endpoint=False)
cam_positions = [np.array([3 * np.cos(a), 1.0, 3 * np.sin(a)]) for a in angles]

train_images, train_origins, train_dirs = [], [], []
for cp in cam_positions:
    img, origins, dirs, hit = analytic_render(cp)
    train_images.append(img); train_origins.append(origins); train_dirs.append(dirs)
train_images = torch.tensor(np.array(train_images))
train_origins = torch.tensor(np.array(train_origins))
train_dirs = torch.tensor(np.array(train_dirs))

# a held-out test view, exactly BETWEEN two training camera positions
test_angle = angles[0] + (angles[1] - angles[0]) / 2
test_cam = np.array([3 * np.cos(test_angle), 1.0, 3 * np.sin(test_angle)])
test_img, test_origins, test_dirs, test_hit = analytic_render(test_cam)
test_img_t, test_origins_t, test_dirs_t = torch.tensor(test_img), torch.tensor(test_origins), torch.tensor(test_dirs)

fig, axes = plt.subplots(1, 5, figsize=(11, 2.5))
for ax, im in zip(axes[:4], train_images[:4]):
    ax.imshow(im.numpy()); ax.axis('off')
axes[4].imshow(test_img); axes[4].axis('off'); axes[4].set_title('held-out\n(never trained on)', fontsize=8)
axes[0].set_title('training views', fontsize=8, loc='left')
plt.show()

## Cảnh như một mạng: tọa độ vào, màu và mật độ ra (The scene as a network: coordinates in, color and density out)
Một NeRF là một MLP nhỏ: đưa vào một điểm 3D, nó trả về một màu và một **mật độ (density)** (cảnh mờ đục đến đâu tại điểm đó — gần 0 trong không gian trống, lớn bên trong vật liệu đặc). Các tọa độ thô `(x, y, z)` trước tiên được mở rộng qua một mã hóa vị trí hình sin ở vài tần số (chính xác là mã hóa vị trí ở Bài 46, áp dụng cho vị trí 3D thay vì vị trí trong chuỗi) — một MLP được đưa vào tọa độ thô ít chiều gặp khó khăn trong việc biểu diễn chi tiết sắc nét, tần số cao, và mã hóa này khắc phục điều đó.

In [ ]:
class TinyNeRF(nn.Module):
    def __init__(self, hidden=64, n_freqs=6):
        super().__init__()
        self.n_freqs = n_freqs
        in_dim = 3 * (2 * n_freqs + 1)
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 4),  # rgb (3) + density (1)
        )
        # bootstrap: bias the initial density prediction upward so early gradients
        # actually reach the color head, instead of vanishing behind near-zero density
        with torch.no_grad():
            self.net[-1].bias[3] = 1.0

    def encode(self, x):
        out = [x]
        for f in range(self.n_freqs):
            out.append(torch.sin(2 ** f * np.pi * x))
            out.append(torch.cos(2 ** f * np.pi * x))
        return torch.cat(out, dim=-1)

    def forward(self, x):
        out = self.net(self.encode(x))
        rgb = torch.sigmoid(out[..., :3])
        sigma = F.softplus(out[..., 3])
        return rgb, sigma

## Kết xuất thể tích: biến các mẫu (màu, mật độ) thành một điểm ảnh (Volumetric rendering: turning (color, density) samples into a pixel)
Với tia của mỗi điểm ảnh, lấy mẫu các điểm dọc theo nó, đánh giá mạng tại mỗi điểm, và tổng hợp các màu được đánh trọng số theo lượng "ánh sáng" của tia còn sống sót để tới được điểm đó — cùng ý tưởng trộn alpha như `cv2.addWeighted` ở Bài 2, nhưng được nối chuỗi qua nhiều mẫu dọc theo một tia thay vì hai ảnh nguyên vẹn: mật độ của mỗi điểm quyết định nó chặn các điểm phía sau bao nhiêu. Toàn bộ phép tính này khả vi từ đầu đến cuối, nên gradient chảy từ sai số của một điểm ảnh được kết xuất ngược về tận trọng số của mạng.

In [ ]:
def render_rays(model, origins, dirs, n_samples=32, near=1.0, far=5.0):
    t_vals = torch.linspace(near, far, n_samples)
    pts = origins[..., None, :] + dirs[..., None, :] * t_vals[:, None]  # (..., n_samples, 3)
    rgb, sigma = model(pts)
    delta = t_vals[1:] - t_vals[:-1]
    delta = torch.cat([delta, torch.tensor([1e10])])
    alpha = 1.0 - torch.exp(-sigma * delta)
    trans = torch.cumprod(torch.cat([torch.ones_like(alpha[..., :1]), 1.0 - alpha + 1e-10], dim=-1), dim=-1)[..., :-1]
    weights = alpha * trans  # how much each sample actually contributes to the final pixel
    return (weights[..., None] * rgb).sum(dim=-2)

torch.manual_seed(0)
model = TinyNeRF()
opt = torch.optim.Adam(model.parameters(), lr=0.005)
V = train_images.shape[0]
origins_flat = train_origins.reshape(V, -1, 3)
dirs_flat = train_dirs.reshape(V, -1, 3)
images_flat = train_images.reshape(V, -1, 3)
for epoch in range(800):
    v = np.random.default_rng(epoch).integers(V)
    idx = np.random.default_rng(epoch + 1000).permutation(origins_flat.shape[1])  # all 256 pixels, shuffled
    o, d, target = origins_flat[v, idx], dirs_flat[v, idx], images_flat[v, idx]
    pred = render_rays(model, o, d)
    loss = F.mse_loss(pred, target)
    opt.zero_grad()
    loss.backward()
    opt.step()

print(f'final training loss: {loss.item():.4f}')

## Tổng hợp góc nhìn mới: kết xuất một vị trí camera chưa từng thấy trong huấn luyện (Novel view synthesis: rendering a camera position never seen in training)
Phép thử thực sự không phải là mạng tái tạo 8 bức ảnh huấn luyện tốt đến đâu — mà là nó có thể kết xuất một vị trí camera *mới* nằm chính xác giữa hai góc nhìn huấn luyện hay không, nơi nó chưa bao giờ nhận được một điểm ảnh giám sát nào.

In [ ]:
with torch.no_grad():
    pred_test = render_rays(model, test_origins_t.reshape(-1, 3), test_dirs_t.reshape(-1, 3)).reshape(IMG, IMG, 3)

mse = F.mse_loss(pred_test, test_img_t).item()
psnr = -10 * np.log10(mse) if mse > 0 else float('inf')
mean_color = train_images.reshape(-1, 3).mean(dim=0)
baseline_mse = F.mse_loss(mean_color[None, None, :].expand(IMG, IMG, 3), test_img_t).item()

print(f'held-out test view MSE:                 {mse:.4f}  (PSNR = {psnr:.1f} dB)')
print(f'baseline (mean training color everywhere) MSE: {baseline_mse:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(test_img); axes[0].set_title('ground truth\n(held-out view)', fontsize=9); axes[0].axis('off')
axes[1].imshow(pred_test.clamp(0, 1).numpy()); axes[1].set_title('NeRF rendering', fontsize=9); axes[1].axis('off')
plt.show()

Góc nhìn mới được kết xuất vượt đường cơ sở màu-trung-bình tầm thường một khoảng rõ ràng, và tái tạo thấy rõ độ đổ bóng và hình bóng của hình cầu từ một góc camera mà nó chưa bao giờ được huấn luyện — mạng đã ngầm học được hình học 3D của hình cầu (nó ở đâu, lớn đến đâu) hoàn toàn từ bằng chứng ảnh chụp 2D và ràng buộc hình học rằng màu của mọi tia phải được giải thích một cách nhất quán trên cả 8 góc nhìn đồng thời.

## 3D Gaussian Splatting: cùng bài toán, một biểu diễn khác (3D Gaussian Splatting: the same problem, a different representation)
**3D Gaussian Splatting** (<a href="../references.html#kerbl-2023-gaussiansplatting">Kerbl et al., 2023</a>) giải quyết bài toán giống hệt — tổng hợp góc nhìn mới dày đặc, chân thực như ảnh chụp từ các ảnh có tư thế đã biết — với một biểu diễn cảnh *tường minh* thay vì MLP ẩn của NeRF: cảnh là một tập hợp lớn các "đốm" Gauss 3D, mỗi đốm có vị trí, kích thước/hướng, màu và độ mờ đục, được tối ưu trực tiếp (không phải truy vấn qua một mạng) để tái tạo các ảnh huấn luyện khi được "vẩy" (splat) (chiếu và trộn alpha, cùng phép toán tổng hợp mà bài này vừa cài đặt) lên mỗi góc nhìn camera. Sự đánh đổi là tốc độ: kết xuất một MLP đòi hỏi một lượt xuôi đầy đủ cho mỗi điểm mẫu dọc theo mọi tia, trong khi kết xuất một tập Gauss cố định là một phép raster hóa (rasterization) rẻ hơn nhiều — gần với đồ họa cổ điển hơn — đó là lý do Gaussian Splatting có thể kết xuất thời gian thực trong khi NeRF về mặt lịch sử thì không. Cả hai cách tiếp cận đều tối ưu hoàn toàn theo hàm mất mát trắc quang 2D, không có dữ liệu chuẩn 3D nào từng có mặt trong vòng lặp; chúng khác nhau ở *thứ gì* được tối ưu, không phải ở *cách* tín hiệu huấn luyện tới được nó.

### Bài tập (Exercises)
1. Giảm `n_train_views` từ 8 xuống 4. PSNR của góc nhìn được giữ lại có giảm đáng kể không — và ảnh được kết xuất có cho thấy các nhiễu thấy rõ ở phía hình cầu giờ nằm xa hơn so với mọi camera huấn luyện không?
2. Đổi `n_samples` trong `render_rays` từ 32 thành 8. Chất lượng kết xuất có suy giảm không, và bạn có thấy tại sao quá ít mẫu dọc theo một tia lại không nắm bắt được bề mặt hình cầu thực sự ở đâu không?
3. Cảnh của bài này là một hình cầu khuếch tán (Lambert) duy nhất, nên mạng chỉ luôn cần học `position -> color, density`, không bao giờ cần `(position, view direction) -> color`. NeRF thực tế còn điều kiện hóa màu theo hướng nhìn, để nắm bắt các hiệu ứng phụ thuộc góc nhìn như điểm phản xạ gương. Hãy phác thảo cách bạn thay đổi `TinyNeRF.forward` để nhận hướng nhìn làm đầu vào thứ hai, và mô tả một vật liệu (ví dụ một hình cầu bóng loáng) mà ở đó điều này tạo ra khác biệt thấy rõ trong khi với hình cầu khuếch tán của bài này thì không.

## Khép lại khóa học (Closing the course)
Bài này khép lại Phần 4, và cùng với nó là cả khóa học: từ điểm ảnh thô và tích chập (Phần 1) qua hình học 3D cổ điển (Phần 2), các đặc trưng tích chập học được (Phần 3), và cơ chế chú ý cùng tự giám sát ở quy mô lớn (Phần 4), kết xuất thần kinh là một điểm kết thúc phù hợp vì nó kết nối toàn bộ hành trình tại một nơi — hình học xạ ảnh và sự hình thành ảnh ở Phần 2, việc học đặc trưng tích chập ở Phần 3, và thói quen của Phần 4 là thay một thuật toán thiết kế thủ công bằng một thuật toán khả vi được huấn luyện hoàn toàn theo các điểm ảnh, tất cả kết hợp lại để tái tạo một cảnh 3D chỉ từ các bức ảnh 2D và ràng buộc rằng chúng phải nhất quán với nhau.